# Python Decorators

### Learning Objectives
- Understand what decorators are and how Python applies them
- Write function decorators with and without parameters
- Use `functools.wraps` to preserve function metadata
- Stack multiple decorators and understand execution order
- Implement class-based decorators
- Apply built-in decorators: `@property`, `@staticmethod`, `@classmethod`
- Build practical decorators: timer, retry, memoize

In [ ]:
import functools
import math
import random
import time

from functools import lru_cache
from IPython.display import HTML, display

## 1. What is a Decorator?

A **decorator** is a function that takes another function as input, extends or modifies its behaviour, and returns a new (wrapped) function — without changing the original function's source code.

**`@` syntax sugar**
```python
@decorator
def func():
    ...
```

Python rewrites this at definition time as:
```python
def func():
    ...
func = decorator(func)   # func is now the wrapper
```

Decorators rely on three Python features:
- **First-class functions** — functions can be passed as arguments and returned from other functions
- **Closures** — inner functions capture variables from their enclosing scope
- **The `@` syntax** — syntactic sugar applied at function definition time

In [ ]:
# Prerequisite: functions are first-class objects in Python

def greet(name):
    return f'Hello, {name}'

# Assign to a variable
say_hello = greet
print(say_hello('Alice'))        # Hello, Alice

# Pass as an argument
def apply(fn, value):
    return fn(value)

print(apply(greet, 'Bob'))       # Hello, Bob

# Return from a function — forms a closure
def make_multiplier(factor):
    def multiply(x):
        return x * factor        # 'factor' captured from enclosing scope
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5), triple(5))      # 10  15

In [ ]:
display(HTML("""
<figure style='margin:16px 0'>
<svg xmlns='http://www.w3.org/2000/svg' width='740' height='200'
     font-family='Segoe UI, Arial, sans-serif'>
  <rect x='0' y='0' width='740' height='200' rx='8' fill='#f8f9fa'/>
  <text x='370' y='28' text-anchor='middle' font-size='16' font-weight='bold' fill='#212529'>Decorator Execution Flow</text>

  <!-- wrapper outer box -->
  <rect x='88' y='48' width='564' height='110' rx='6' fill='#e8f4f8' stroke='#2166ac' stroke-width='2'/>
  <text x='100' y='66' font-size='12' font-weight='bold' fill='#2166ac'>wrapper</text>

  <!-- pre-work box -->
  <rect x='108' y='70' width='130' height='70' rx='4' fill='#cce5ff' stroke='#2166ac' stroke-width='1.5'/>
  <text x='173' y='109' text-anchor='middle' font-size='13' font-weight='bold' fill='#2166ac'>pre-work</text>

  <!-- original func box -->
  <rect x='285' y='70' width='170' height='70' rx='4' fill='#d4edda' stroke='#28a745' stroke-width='1.5'/>
  <text x='370' y='102' text-anchor='middle' font-size='13' font-weight='bold' fill='#28a745'>original func</text>
  <text x='370' y='122' text-anchor='middle' font-size='12' fill='#28a745'>(*args, **kwargs)</text>

  <!-- post-work box -->
  <rect x='502' y='70' width='130' height='70' rx='4' fill='#cce5ff' stroke='#2166ac' stroke-width='1.5'/>
  <text x='567' y='109' text-anchor='middle' font-size='13' font-weight='bold' fill='#2166ac'>post-work</text>

  <!-- arrow pre-work -> func -->
  <line x1='238' y1='105' x2='280' y2='105' stroke='#495057' stroke-width='2'/>
  <polygon points='278,101 286,105 278,109' fill='#495057'/>

  <!-- arrow func -> post-work -->
  <line x1='455' y1='105' x2='497' y2='105' stroke='#495057' stroke-width='2'/>
  <polygon points='495,101 503,105 495,109' fill='#495057'/>

  <!-- args arrow into wrapper -->
  <line x1='10' y1='105' x2='82' y2='105' stroke='#495057' stroke-width='2'/>
  <polygon points='80,101 88,105 80,109' fill='#495057'/>
  <text x='49' y='97' text-anchor='middle' font-size='12' fill='#495057'>args</text>

  <!-- result arrow out of wrapper -->
  <line x1='652' y1='105' x2='726' y2='105' stroke='#495057' stroke-width='2'/>
  <polygon points='724,101 732,105 724,109' fill='#495057'/>
  <text x='689' y='97' text-anchor='middle' font-size='12' fill='#495057'>result</text>

  <!-- returns wrapper label -->
  <text x='370' y='182' text-anchor='middle' font-size='12' fill='#6c757d'>decorator(func) &#8594; returns wrapper &#8212; wrapper is called in place of the original</text>
</svg>
<figcaption style='font-size:12px;color:#6c757d;text-align:center;margin-top:4px'>The wrapper intercepts the call, adds behaviour, then invokes the original function</figcaption>
</figure>
"""))

## 2. Basic Decorator

In [ ]:
# Without functools.wraps — metadata is lost
def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout
def greet(name):
    """Return a personalised greeting."""
    return f'hello, {name}'

print(greet('Alice'))        # HELLO, ALICE
print(greet.__name__)        # wrapper  <- original name is lost
print(greet.__doc__)         # None     <- docstring is gone

In [ ]:
def shout(func):
    @functools.wraps(func)       # copies __name__, __doc__, __module__, __qualname__
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout
def greet(name):
    """Return a personalised greeting."""
    return f'hello, {name}'

print(greet('Alice'))        # HELLO, ALICE
print(greet.__name__)        # greet   <- preserved
print(greet.__doc__)         # Return a personalised greeting.

# The @ syntax is identical to manual wrapping:
def greet2(name):
    return f'hello, {name}'

greet2 = shout(greet2)       # same result as @shout
print(greet2('Bob'))         # HELLO, BOB

## 3. Decorators with Parameters (Decorator Factory)

When a decorator needs configuration, add an outer function that accepts the parameters and returns the actual decorator. This creates three layers:

```python
@repeat(n=3)           # outer call with parameters
def task():
    ...
```

Python applies this as:
```python
task = repeat(n=3)(task)   # repeat(n=3) returns a decorator; that decorator wraps task
```

| Layer | Role |
|-------|------|
| **factory** `repeat(n)` | Accepts parameters, returns `decorator` |
| **decorator** `decorator(func)` | Accepts `func`, returns `wrapper` |
| **wrapper** `wrapper(*args)` | Runs around every call to `func` |

In [ ]:
def repeat(n=1):
    """Call the decorated function n times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(n=3)
def say(msg):
    print(msg)

say('hello')

# Parameterised logging decorator
def log_if_slow(threshold_ms=100):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            t0 = time.perf_counter()
            result = func(*args, **kwargs)
            elapsed = (time.perf_counter() - t0) * 1000
            if elapsed > threshold_ms:
                print(f'[SLOW] {func.__name__} took {elapsed:.1f} ms')
            return result
        return wrapper
    return decorator

@log_if_slow(threshold_ms=1)
def fast(): pass

@log_if_slow(threshold_ms=1)
def slow():
    time.sleep(0.01)

print()
fast()   # no output — under threshold
slow()   # [SLOW] slow took ~10 ms

## 4. Stacking Decorators

Multiple decorators can be stacked. They are applied **bottom-up** at definition time but execute **top-down** at call time.

```python
@A
@B
def f():
    ...
```

Equivalent to: `f = A(B(f))`
- `B` wraps `f` first (innermost)
- `A` wraps the result (outermost)
- Call order: `A-pre → B-pre → f → B-post → A-post`

In [ ]:
display(HTML("""
<figure style='margin:16px 0'>
<svg xmlns='http://www.w3.org/2000/svg' width='740' height='290'
     font-family='Segoe UI, Arial, sans-serif'>
  <rect x='0' y='0' width='740' height='290' rx='8' fill='#f8f9fa'/>
  <text x='370' y='30' text-anchor='middle' font-size='16' font-weight='bold' fill='#212529'>Decorator Stacking: @A  @B  def f()</text>

  <!-- Left: source code panel -->
  <rect x='18' y='48' width='300' height='220' rx='6' fill='white' stroke='#dee2e6' stroke-width='1.5'/>
  <text x='168' y='70' text-anchor='middle' font-size='13' font-weight='bold' fill='#6c757d'>Written as</text>
  <line x1='18' y1='78' x2='318' y2='78' stroke='#dee2e6' stroke-width='1'/>

  <rect x='34' y='92' width='60' height='28' rx='3' fill='#ffeeba' stroke='#d39e00' stroke-width='1'/>
  <text x='64' y='111' text-anchor='middle' font-size='13' font-family='monospace' font-weight='bold' fill='#856404'>@A</text>
  <text x='110' y='111' font-size='11' fill='#6c757d'>&#8592; applied last (outer)</text>

  <rect x='34' y='130' width='60' height='28' rx='3' fill='#cce5ff' stroke='#2166ac' stroke-width='1'/>
  <text x='64' y='149' text-anchor='middle' font-size='13' font-family='monospace' font-weight='bold' fill='#2166ac'>@B</text>
  <text x='110' y='149' font-size='11' fill='#6c757d'>&#8592; applied first (inner)</text>

  <text x='34' y='185' font-size='13' font-family='monospace' fill='#212529'>def f():</text>
  <text x='50' y='205' font-size='13' font-family='monospace' fill='#212529'>...</text>

  <text x='168' y='248' text-anchor='middle' font-size='12' fill='#495057'>f = A(B(f))</text>

  <!-- Right: execution order panel -->
  <rect x='338' y='48' width='384' height='220' rx='6' fill='white' stroke='#dee2e6' stroke-width='1.5'/>
  <text x='530' y='70' text-anchor='middle' font-size='13' font-weight='bold' fill='#6c757d'>Call order when f() is invoked</text>
  <line x1='338' y1='78' x2='722' y2='78' stroke='#dee2e6' stroke-width='1'/>

  <!-- Nested boxes -->
  <rect x='355' y='88' width='350' height='160' rx='5' fill='#fff8e1' stroke='#d39e00' stroke-width='2'/>
  <text x='368' y='106' font-size='12' font-weight='bold' fill='#856404'>A wrapper</text>

  <rect x='372' y='114' width='316' height='110' rx='5' fill='#e8f4f8' stroke='#2166ac' stroke-width='2'/>
  <text x='385' y='131' font-size='12' font-weight='bold' fill='#2166ac'>B wrapper</text>

  <rect x='389' y='140' width='282' height='60' rx='4' fill='#d4edda' stroke='#28a745' stroke-width='1.5'/>
  <text x='530' y='166' text-anchor='middle' font-size='13' font-weight='bold' fill='#28a745'>original f(*args)</text>

  <!-- call order arrow -->
  <text x='530' y='265' text-anchor='middle' font-size='11' fill='#6c757d'>A-pre &#8594; B-pre &#8594; f &#8594; B-post &#8594; A-post</text>
</svg>
<figcaption style='font-size:12px;color:#6c757d;text-align:center;margin-top:4px'>Bottom-up wrapping, top-down execution</figcaption>
</figure>
"""))

In [ ]:
def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f'<b>{func(*args, **kwargs)}</b>'
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f'<i>{func(*args, **kwargs)}</i>'
    return wrapper

@bold          # applied last — outermost wrapper
@italic        # applied first — innermost wrapper
def greet(name):
    return f'Hello, {name}'

print(greet('Alice'))   # <b><i>Hello, Alice</i></b>

# Equivalent manual application:
def greet2(name):
    return f'Hello, {name}'

greet2 = bold(italic(greet2))
print(greet2('Bob'))    # <b><i>Hello, Bob</i></b>

## 5. Class-Based Decorators

A class can act as a decorator if it implements `__init__` (receives the function) and `__call__` (runs when the wrapped function is invoked).

Use a class-based decorator when you need **state** across calls (e.g., call counts, cache).

```python
class MyDecorator:
    def __init__(self, func):    # receives the original function
        functools.update_wrapper(self, func)
        self.func = func

    def __call__(self, *args, **kwargs):   # called in place of func
        # pre-work
        result = self.func(*args, **kwargs)
        # post-work
        return result
```

In [ ]:
class CountCalls:
    """Track how many times the decorated function has been called."""

    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.num_calls = 0

    def __call__(self, *args, **kwargs):
        self.num_calls += 1
        print(f'Call #{self.num_calls} to {self.func.__name__}')
        return self.func(*args, **kwargs)

@CountCalls
def add(a, b):
    return a + b

print(add(1, 2))                 # Call #1 to add  →  3
print(add(3, 4))                 # Call #2 to add  →  7
print(f'Total calls: {add.num_calls}')   # Total calls: 2
print(add.__name__)              # add  (update_wrapper preserved it)

## 6. Built-in Decorators

Python provides three built-in decorators for class methods:

| Decorator | Receives | Use case |
|-----------|----------|----------|
| `@property` | `self` | Expose a method as a read-only attribute |
| `@staticmethod` | nothing | Utility function logically grouped with the class |
| `@classmethod` | `cls` | Alternative constructor or factory method |

In [ ]:
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):
        return self._radius

    @property
    def area(self):
        return math.pi * self._radius ** 2

    @staticmethod
    def is_valid_radius(r):
        return r > 0

    @classmethod
    def unit(cls):
        return cls(radius=1)    # alternative constructor

    def __repr__(self):
        return f'Circle(radius={self._radius})'

c = Circle(5)
print(c.radius)                      # 5       (no parentheses — it's a property)
print(f'{c.area:.4f}')               # 78.5398
print(Circle.is_valid_radius(-1))    # False
print(Circle.unit())                 # Circle(radius=1)

## 7. Practical Decorators

In [ ]:
def timer(func):
    """Print execution time of the decorated function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0     = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        print(f'[timer] {func.__name__} finished in {elapsed:.4f}s')
        return result
    return wrapper

@timer
def compute(n):
    return sum(x**2 for x in range(n))

result = compute(500_000)
print(f'result = {result:,}')

In [ ]:
def retry(max_attempts=3, delay=0.1, exceptions=(Exception,)):
    """Retry a failing function up to max_attempts times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    print(f'[retry] attempt {attempt}/{max_attempts} failed: {exc}')
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise RuntimeError(f'{func.__name__} failed after {max_attempts} attempts')
        return wrapper
    return decorator

@retry(max_attempts=4, delay=0.0, exceptions=(ValueError,))
def flaky(counter=[0]):
    counter[0] += 1
    if counter[0] < 4:
        raise ValueError(f'not ready (call {counter[0]})')
    return 'success'

print(flaky())

In [ ]:
# Manual memoize decorator
def memoize(func):
    """Cache results keyed by arguments."""
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache    # expose cache for inspection
    return wrapper

@memoize
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print([fib(i) for i in range(10)])
print(f'cache entries: {len(fib.cache)}')

# Python stdlib provides this as functools.lru_cache
@lru_cache(maxsize=128)
def fib2(n):
    return n if n < 2 else fib2(n - 1) + fib2(n - 2)

print(fib2(30))
print(fib2.cache_info())

## 8. Advantages

| Advantage | Detail |
|-----------|--------|
| **Separation of concerns** | Cross-cutting concerns (logging, timing, auth) are kept out of business logic |
| **Reusability** | One decorator can be applied to many functions without duplicating code |
| **Readable intent** | `@login_required`, `@retry(3)` communicates behaviour at a glance |
| **Non-invasive** | The original function's source is untouched; decoration is applied at use-site |
| **Composable** | Multiple decorators stack cleanly; each layer is independent |
| **DRY** | Eliminates boilerplate that would otherwise be copied into every function |

## 9. Disadvantages

| Disadvantage | Detail |
|-------------|--------|
| **Debugging difficulty** | Stack traces show `wrapper` instead of the original function unless `functools.wraps` is used |
| **Hidden behaviour** | A reader must look up the decorator definition to understand what a function really does |
| **Stacking order is subtle** | `@A @B` vs `@B @A` produce different results; easy to get wrong |
| **Performance overhead** | Each decorated call adds a function frame; negligible in most cases but relevant in tight loops |
| **Class decoration is awkward** | Decorating class methods (especially `@classmethod` or `@staticmethod`) requires extra care |
| **Testing complexity** | Testing decorated functions sometimes requires unwrapping via `func.__wrapped__` |

In [ ]:
# 1. Without functools.wraps — traceback is misleading
def bad_decorator(func):
    def wrapper(*args, **kwargs):        # __name__ is 'wrapper'
        return func(*args, **kwargs)
    return wrapper

def good_decorator(func):
    @functools.wraps(func)               # __name__ is preserved
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def calc_bad(): pass

@good_decorator
def calc_good(): pass

print('bad  __name__:', calc_bad.__name__)    # wrapper
print('good __name__:', calc_good.__name__)   # calc_good

# 2. Access the unwrapped function for testing
print('wrapped under good_decorator:', calc_good.__wrapped__)

# 3. Stacking order pitfall
def add_prefix(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return 'PREFIX-' + func(*args, **kwargs)
    return wrapper

def add_suffix(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) + '-SUFFIX'
    return wrapper

@add_prefix
@add_suffix
def label():
    return 'CORE'

print(label())   # PREFIX-CORE-SUFFIX  (suffix applied first, then prefix)

## 10. Summary

| Pattern | Syntax | When to use |
|---------|--------|-------------|
| Basic decorator | `def dec(func): ... return wrapper` | Simple pre/post behaviour |
| With `functools.wraps` | `@functools.wraps(func)` inside wrapper | Always — preserves metadata |
| Decorator factory | `def dec(param): def decorator(func): ...` | Configurable decorators |
| Stacked decorators | `@A` above `@B` | Combine independent concerns |
| Class-based | `__init__` + `__call__` | When state across calls is needed |
| `@property` | Built-in | Attribute-style access to computed values |
| `@staticmethod` | Built-in | Utilities grouped with a class but needing no `self`/`cls` |
| `@classmethod` | Built-in | Alternative constructors / factory methods |
| `@lru_cache` | `functools` | Memoisation with an LRU eviction policy |

### Quick decision guide

- Need to add logging, timing, auth, or retry to many functions? **→ function decorator**
- Need the decorator to accept configuration? **→ decorator factory**
- Need to count calls or cache state between invocations? **→ class-based decorator**
- Need memoisation for a pure function? **→ `@functools.lru_cache`**

## 11. All Decorator Patterns — Side by Side

All six patterns below implement the same `logger` behaviour (print before and after the call) so you can compare structure, not logic.

| # | Approach | Shape | `@dec` (no parens) | `@dec(param)` (with parens) | Per-function state |
|---|----------|-------|--------------------|-----------------------------|--------------------|
| 1 | Function — no params | `def dec(func)` | ✓ | ✗ | ✗ |
| 2 | Function factory | `def dec(param)` → `def decorator(func)` | ✗ | ✓ | ✗ |
| 3 | Function — optional params | `def dec(func=None, *, param)` | ✓ | ✓ | ✗ |
| 4 | Class — no params | `__init__(func)` + `__call__` as wrapper | ✓ | ✗ | ✓ |
| 5 | Class — with params | `__init__(param)` + `__call__(func)` → wrapper | ✗ | ✓ | ✓ (shared) |
| 6 | Class — optional params | `__init__(func=None, *, param)` + dispatch in `__call__` | ✓ | ✓ | ✓ |

In [ ]:
# ─── Pattern 1: Function decorator — no parameters ───────────────────────────
# The simplest shape: one function that takes another function and returns a wrapper.
# Python does: add = logger(add)
# Use as: @logger  (no parentheses, ever)

def logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f'[LOG] calling {func.__name__}{args}')
        result = func(*args, **kwargs)
        print(f'[LOG] done    {func.__name__} -> {result}')
        return result
    return wrapper

@logger
def add(a, b):
    return a + b

add(2, 3)

In [ ]:
# ─── Pattern 2: Function decorator factory — with parameters ──────────────────
# Three nested functions: factory → decorator → wrapper.
# The factory accepts config and returns the decorator.
# Python does: add = logger(prefix='INFO')(add)
# Must always use parentheses: @logger(prefix='INFO'), never @logger alone.

def logger(prefix='LOG'):              # layer 1 — factory, accepts config
    def decorator(func):               # layer 2 — accepts the function
        @functools.wraps(func)
        def wrapper(*args, **kwargs):  # layer 3 — runs on every call
            print(f'[{prefix}] calling {func.__name__}{args}')
            result = func(*args, **kwargs)
            print(f'[{prefix}] done    {func.__name__} -> {result}')
            return result
        return wrapper
    return decorator

@logger(prefix='INFO')
def add(a, b):
    return a + b

@logger()              # parens required even with defaults
def mul(a, b):
    return a * b

add(2, 3)
mul(4, 5)

In [ ]:
# ─── Pattern 3: Function decorator — optional parameters ─────────────────────
# Works as BOTH @logger (no parens) and @logger(prefix='X') (with parens).
# Trick: declare func as the first positional param with a default of None,
#   and all config params as keyword-only (after *).
# If func is not None → called without parens; wrap immediately and return.
# If func is None     → called with parens; return the decorator for the next call.

def logger(func=None, *, prefix='LOG'):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            print(f'[{prefix}] calling {fn.__name__}{args}')
            result = fn(*args, **kwargs)
            print(f'[{prefix}] done    {fn.__name__} -> {result}')
            return result
        return wrapper
    if func is not None:
        return decorator(func)   # @logger — no parens, func is the decorated fn
    return decorator             # @logger(...) — return decorator to be called next

@logger                    # no parentheses — func receives 'add' directly
def add(a, b):
    return a + b

@logger(prefix='DEBUG')    # with parentheses — func=None, decorator returned first
def sub(a, b):
    return a - b

add(2, 3)
sub(5, 2)

In [ ]:
# ─── Pattern 4: Class decorator — no parameters ──────────────────────────────
# __init__ receives the function; __call__ acts as the wrapper on every invocation.
# Python does: add = Logger(add)  — the Logger instance replaces the function name.
# Advantage over a function decorator: instance variables give persistent state
#   per decorated function (call count, cache, last result, etc.).

class Logger:
    def __init__(self, func):
        functools.update_wrapper(self, func)   # copies __name__, __doc__, etc.
        self.func = func
        self.call_count = 0

    def __call__(self, *args, **kwargs):
        self.call_count += 1
        print(f'[LOG] calling {self.func.__name__}{args}  (call #{self.call_count})')
        result = self.func(*args, **kwargs)
        print(f'[LOG] done    {self.func.__name__} -> {result}')
        return result

@Logger
def add(a, b):
    return a + b

add(2, 3)
add(10, 20)
print(f'add was called {add.call_count} times')
print(f'add.__name__ = {add.__name__}')   # preserved by update_wrapper

In [ ]:
# ─── Pattern 5: Class decorator — with parameters ────────────────────────────
# __init__ receives parameters; __call__ receives the function and returns a wrapper.
# Python does: add = Logger(prefix='INFO')(add)
# Two-step: Logger(prefix='INFO') creates the instance, then instance(add) wraps it.
# Advantage: the instance can be reused across multiple decorated functions,
#            accumulating shared state (e.g. total call count across all functions).

class Logger:
    def __init__(self, prefix='LOG'):
        self.prefix = prefix
        self.call_count = 0

    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            self.call_count += 1
            print(f'[{self.prefix}] calling {func.__name__}{args}  (call #{self.call_count})')
            result = func(*args, **kwargs)
            print(f'[{self.prefix}] done    {func.__name__} -> {result}')
            return result
        return wrapper

info_logger = Logger(prefix='INFO')   # one instance, shared by both functions

@info_logger
def add(a, b):
    return a + b

@info_logger
def sub(a, b):
    return a - b

add(2, 3)
sub(5, 2)
add(10, 1)
print(f'info_logger.call_count across both functions = {info_logger.call_count}')

In [ ]:
# ─── Pattern 6: Class decorator — optional parameters ────────────────────────
# Detects in __init__ whether it received a callable (used as @Logger, no parens)
# or a keyword (used as @Logger(prefix='X')).
# When @Logger:          func is set in __init__; __call__ acts as the wrapper.
# When @Logger(prefix):  func is None; __call__(func) returns a wrapper function.

class Logger:
    def __init__(self, func=None, *, prefix='LOG'):
        self.prefix = prefix
        self.call_count = 0
        self.func = func
        if func is not None:
            functools.update_wrapper(self, func)

    def _make_wrapper(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            self.call_count += 1
            print(f'[{self.prefix}] calling {func.__name__}{args}  (call #{self.call_count})')
            result = func(*args, **kwargs)
            print(f'[{self.prefix}] done    {func.__name__} -> {result}')
            return result
        return wrapper

    def __call__(self, *args, **kwargs):
        if self.func is None:
            # @Logger(prefix='X') path — first __call__ receives the function
            return self._make_wrapper(args[0])
        # @Logger path — act directly as the wrapper
        self.call_count += 1
        print(f'[{self.prefix}] calling {self.func.__name__}{args}  (call #{self.call_count})')
        result = self.func(*args, **kwargs)
        print(f'[{self.prefix}] done    {self.func.__name__} -> {result}')
        return result

@Logger                      # no parentheses — func set in __init__
def add(a, b):
    return a + b

@Logger(prefix='DEBUG')      # with parentheses — func=None in __init__
def sub(a, b):
    return a - b

add(2, 3)
sub(5, 2)
print(f'add.call_count = {add.call_count}')